GRM / Genotype Preparation
Prepare a high-quality, pruned genotype set for Step-1 ridge regression.

This ensures that only common, well-QC’d variants with consistent IDs are retained,
providing REGENIE with a clean, reliable input for building genome-wide polygenic predictors.

In [1]:
# Import required libraries
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

# Ensuring dsub is up to date
#!pip3 install --upgrade dsub

## Load in Variables

In [2]:
%run /home/jupyter/workspace/workspace-scripts/Var_set_2.ipynb

bucket_input:   gs://workspace-bucket-wb-blinding-cabbage-2295
bucket_output:  gs://workspace-output-wb-blinding-cabbage-2295
bucket_scripts: gs://script-wb-blinding-cabbage-2295
bucket_GenDat: gs://workspace-gendat-wb-blinding-cabbage-2295
dataset:        wb-silky-artichoke-2408.C2024Q3R8


In [3]:
# Save this Python variable as an environment variable so that its easier to use within %%bash cells
%env JOB_ID={LINE_COUNT_JOB_ID}
#my_bucket = os.environ['WORKSPACE_BUCKET']
pd.set_option('display.max_colwidth', 0)

results='grm' # Directory for results for dsu
%env results={results}

JOB_NAME='clinvar_v8' # Job Name for dsu
%env JOB_NAME={JOB_NAME}

USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')
%env USER_NAME={USER_NAME}

# ANCESTRY="EUR"
# %env ANCESTRY={ANCESTRY}

env: JOB_ID={LINE_COUNT_JOB_ID}
env: results=grm
env: JOB_NAME=clinvar_v8
env: USER_NAME='nathani


In [4]:
# Analysis Results Folder
output_folder = os.path.join(
    bucket_GenDat,
    results,
    JOB_NAME
)
%env output_folder={output_folder}

output_logs_folder = os.path.join(output_folder, "logs") # Log files folder
%env output_logs_folder={output_logs_folder}

env: output_folder=gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8
env: output_logs_folder=gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/logs


# **PLINK format**

## Pruning

### make a script for pruning, MAF, Geno, hwe and autosome filtering

In [5]:
%%writefile /home/jupyter/workspace/workspace-scripts/dsub_Plink_auto.sh

set -o pipefail 
set -o errexit

plink2 \
    --bfile "${bed_file%.bed}" \
    --chr 1-22 \
    --maf 0.01 \
    --geno 0.1 \
    --hwe 1e-6 \
    --indep-pairwise 50 5 0.2 \
    --threads 7 \
    --memory 30000 \
    --make-bed \
    --out "${output_folder}/autosomes_pruned"


echo "output_folder: ${output_folder}"


Overwriting /home/jupyter/workspace/workspace-scripts/dsub_Plink_auto.sh


In [11]:
!echo ${GOOGLE_PROJECT}

'wb-blinding-cabbage-2295'


In [26]:
%%bash --out test
#########################################
#Tutorial Script
#########################################
dsub \
    --provider google-batch \
    --user-project wb-blinding-cabbage-2295 \
    --project wb-blinding-cabbage-2295 \
    --network "global/networks/network" \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --use-private-address \
    --user "${DSUB_USER_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --name "${JOB_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/biocontainer/plink2:alpha2.3_jan2020" \
    --machine-type "n2-highmem-2" \
    --timeout "14d" \
    --boot-disk-size 20 \
    --output-recursive out_path="${output_folder}" \
    --command 'echo hello world > ${out_path}/test11.txt && \
             plink2 --help > ${out_path}/help_plink.txt && \
             nproc > ${out_path}/vm_cpu.txt && \
             cat /proc/meminfo | grep MemTotal > ${out_path}/vm_ram.txt && \
             df -h > ${out_path}/vm_disk.txt'

Job properties:
  job-id: clinvar-v8--jupyter--260629-053349-26
  job-name: clinvar-v8
  user-id: jupyter
Provider internal-id (operation): projects/wb-blinding-cabbage-2295/locations/us-central1/jobs/clinvar-v8--jupyter--260629-053349-26-0-0
Launched job-id: clinvar-v8--jupyter--260629-053349-26
To check the status, run:
  dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260629-053349-26' --users 'jupyter' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260629-053349-26' --users 'jupyter'


In [27]:
  !dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260629-053349-26' --users 'jupyter' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
clinvar-v8  SUCCESS   06-29 05:35:46



In [44]:
!echo "${OWNER_EMAIL}" | cut -d@ -f1 | tr '.' '-' | tr -d '\r'

'nathani


In [19]:
JOB_NAME='clinvar_v8' # Job Name for dsub
%env JOB_NAME={JOB_NAME}

env: JOB_NAME=clinvar_v8


In [45]:
%%bash --out filtering

# Get a shorter username to leave more characters for the job name
DSUB_USER_NAME=nathani

MACHINE_TYPE="n2-standard-8"
#MACHINE_TYPE="n2-standard-16"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
# Change for your bucket, path in output of cell directly above:
#BASH_SCRIPT="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_auto.sh"
BASH_SCRIPT="gs://script-wb-blinding-cabbage-2295/dsub_Plink_auto.sh"
#--image "gcr.io/bick-aps2/plink2:polygenic" \
#google-cls-v2
dsub \
    --provider google-batch \
    --user-project wb-blinding-cabbage-2295 \
    --project wb-blinding-cabbage-2295 \
    --network "global/networks/network" \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --use-private-address \
    --user "${DSUB_USER_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --name "${JOB_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/biocontainer/plink2:alpha2.3_jan2020" \
    --preemptible \
    --boot-disk-size 100 \
    --disk-size 200 \
    --timeout "2d" \
    --machine-type ${MACHINE_TYPE} \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=wb-blinding-cabbage-2295 \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bim_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input fam_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --output-recursive output_folder="${output_folder}"

httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Job properties:
  job-id: clinvar-v8--nathani--260629-062026-39
  job-name: clinvar-v8
  user-id: nathani
Provider internal-id (operation): projects/wb-blinding-cabbage-2295/locations/us-central1/jobs/clinvar-v8--nathani--260629-062026-39-0-0
Launched job-id: clinvar-v8--nathani--260629-062026-39
To check the status, run:
  dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260629-062026-39' --users 'nathani' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260629-062026-39' --users 'nathani'


In [48]:
!dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260629-062026-39' --users 'nathani' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
clinvar-v8  RUNNING   06-29 06:21:53



## extract pruned SNPs

In [11]:
%%writefile /home/jupyter/workspace/workspace-scripts/dsub_Plink_extract.sh

set -o pipefail 
set -o errexit


plink2 \
    --bfile "${bed_file%.bed}" \
    --extract "${in_file}" \
    --threads 3 \
    --memory 12000 \
    --make-bed \
    --out "${output_folder}/autosomes_final_pruned"


Overwriting /home/jupyter/workspace/workspace-scripts/dsub_Plink_extract.sh


In [ ]:
JOB_NAME='extractin_pruned_SNPs' # Job Name for dsub
%env JOB_NAME={JOB_NAME}

In [13]:
%%bash --out extracting

# Get a shorter username to leave more characters for the job name
#DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"
BSUB_USER_NAME=nathani

MACHINE_TYPE="n2-standard-4"
#MACHINE_TYPE="n2-standard-16"


TIMESTAMP=$(date +"%Y%m%d_%H%M%S")

# Change for your bucket, path in output of cell directly above:
BASH_SCRIPT="${WORKSPACE_BUCKET_SCRIPTS}/dsub_Plink_extract.sh"

#clear log folder
#gsutil -m rm "${output_logs_folder}/**"

#--image "gcr.io/bick-aps2/plink2:polygenic" \
#google-cls-v2
    dsub \
    --provider google-batch \
    --user-project wb-blinding-cabbage-2295 \
    --project wb-blinding-cabbage-2295 \
    --network "global/networks/network" \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --use-private-address \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --name "${JOB_NAME}" \
    --preemptible \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/biocontainer/plink2:alpha2.3_jan2020" \
    --boot-disk-size 100 \
    --disk-size 300 \
    --machine-type ${MACHINE_TYPE} \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=wb-blinding-cabbage-2295 \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/autosomes_pruned.bed" \
    --input bim_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/autosomes_pruned.bim" \
    --input fam_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/autosomes_pruned.fam" \
    --input in_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/autosomes_pruned.prune.in" \
    --output-recursive output_folder="${output_folder}"

httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Job properties:
  job-id: clinvar-v8--jupyter--260630-061823-18
  job-name: clinvar-v8
  user-id: jupyter
Provider internal-id (operation): projects/wb-blinding-cabbage-2295/locations/us-central1/jobs/clinvar-v8--jupyter--260630-061823-18-0-0
Launched job-id: clinvar-v8--jupyter--260630-061823-18
To check the status, run:
  dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260630-061823-18' --users 'jupyter' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260630-061823-18' --users 'jupyter'


In [21]:
!dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260630-061823-18' --users 'jupyter' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
clinvar-v8  SUCCESS   06-30 06:27:23



In [20]:
! wc -l /home/jupyter/workspace/workspace-GenDat/grm/clinvar_v8/autosomes_final_pruned.bim
! wc -l /home/jupyter/workspace/workspace-GenDat/grm/clinvar_v8/autosomes_pruned.bim

74906 /home/jupyter/workspace/workspace-GenDat/grm/clinvar_v8/autosomes_final_pruned.bim
102101 /home/jupyter/workspace/workspace-GenDat/grm/clinvar_v8/autosomes_pruned.bim


## Now that the plink files have been made for the GRM, I need to change the FID column to match the pheno file and the BGEN files

In [4]:
!head -20 /home/jupyter/workspace/workspace-GenDat/grm/clinvar_v8/autosomes_final_pruned.bim

1	chr1:903175:C:A	0	903175	A	C
1	chr1:903285:T:C	0	903285	C	T
1	chr1:905373:T:C	0	905373	C	T
1	chr1:908920:C:G	0	908920	G	C
1	chr1:979230:C:T	0	979230	T	C
1	chr1:984475:G:A	0	984475	A	G
1	chr1:996120:C:T	0	996120	T	C
1	chr1:1135087:G:A	0	1135087	A	G
1	chr1:1290683:G:A	0	1290683	A	G
1	chr1:1292946:A:T	0	1292946	T	A
1	chr1:1335607:G:A	0	1335607	A	G
1	chr1:1338494:C:T	0	1338494	T	C
1	chr1:1374025:T:C	0	1374025	C	T
1	chr1:1379327:A:G	0	1379327	G	A
1	chr1:1430379:G:A	0	1430379	A	G
1	chr1:1531942:T:G	0	1531942	G	T
1	chr1:1555168:A:G	0	1555168	G	A
1	chr1:1567493:A:G	0	1567493	G	A
1	chr1:1569658:A:G	0	1569658	G	A
1	chr1:1776301:T:G	0	1776301	G	T


In [7]:
!head -20 /home/jupyter/workspace/workspace-GenDat/grm/clinvar_v8/autosomes_final_pruned.fam

0	1000000	0	0	0	-9
0	1000004	0	0	0	-9
0	1000033	0	0	0	-9
0	1000039	0	0	0	-9
0	1000042	0	0	0	-9
0	1000045	0	0	0	-9
0	1000059	0	0	0	-9
0	1000061	0	0	0	-9
0	1000062	0	0	0	-9
0	1000070	0	0	0	-9
0	1000075	0	0	0	-9
0	1000079	0	0	0	-9
0	1000091	0	0	0	-9
0	1000093	0	0	0	-9
0	1000095	0	0	0	-9
0	1000102	0	0	0	-9
0	1000104	0	0	0	-9
0	1000105	0	0	0	-9
0	1000109	0	0	0	-9
0	1000112	0	0	0	-9


In [9]:
!head ~/workspace/workspace-input/phenotype_European_women.txt

3496558	3496558	0
8071156	8071156	0
1560790	1560790	0
8094101	8094101	0
1584564	1584564	0
5270867	5270867	0
2358852	2358852	0
2650858	2650858	0
1637236	1637236	0
2705449	2705449	0


In [14]:
fam_file= pd.read_csv("~/workspace/workspace-GenDat/grm/clinvar_v8/autosomes_final_pruned.fam", sep = '\t', header=None)
fam_file.iloc[:,0] = fam_file.iloc[:,1]
fam_file
fam_file.to_csv(f'{bucket_GenDat}/grm/clinvar_v8/updated_FID.fam', sep='\t', header=None, index=False)


In [16]:
!head ~/workspace/workspace-GenDat/grm/clinvar_v8/updated_FID.fam

1000000	1000000	0	0	0	-9
1000004	1000004	0	0	0	-9
1000033	1000033	0	0	0	-9
1000039	1000039	0	0	0	-9
1000042	1000042	0	0	0	-9
1000045	1000045	0	0	0	-9
1000059	1000059	0	0	0	-9
1000061	1000061	0	0	0	-9
1000062	1000062	0	0	0	-9
1000070	1000070	0	0	0	-9


## Now write a plink script to incorporate these FIDs into the existing fam file

In [10]:
%%writefile ~/workspace/workspace-scripts/Plink_update_FID.sh

set -o pipefail 
set -o errexit

plink2 \
    --bfile "${bed_file%.bed}" \
    --fam "${in_file}" \
    --threads 3 \
    --memory 12000 \
    --make-bed \
    --out "${output_folder}/autosomes_final_pruned_updated_FID"

Overwriting /home/jupyter/workspace/workspace-scripts/Plink_update_FID.sh


In [20]:
JOB_NAME='update_FID' # Job Name for dsu
%env JOB_NAME={JOB_NAME}

env: JOB_NAME=update_FID


In [11]:
%%bash --out fam_merge
# Get a shorter username to leave more characters for the job name
#DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"
DSUB_USER_NAME=nathani

MACHINE_TYPE="n2-standard-4"
#MACHINE_TYPE="n2-standard-16"

TIMESTAMP=$(date +"%Y%m%d_%H%M%S")

# Change for your bucket, path in output of cell directly above:
BASH_SCRIPT="${WORKSPACE_BUCKET_SCRIPTS}/Plink_update_FID.sh"

#clear log folder
#gsutil -m rm "${output_logs_folder}/**"

#--image "gcr.io/bick-aps2/plink2:polygenic" \
#google-cls-v2
    dsub \
    --provider google-batch \
    --user-project wb-blinding-cabbage-2295 \
    --project wb-blinding-cabbage-2295 \
    --network "global/networks/network" \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --use-private-address \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-${JOB_NAME}.log" \
    "$@" \
    --name "${JOB_NAME}" \
    --preemptible \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/biocontainer/plink2:alpha2.3_jan2020" \
    --boot-disk-size 100 \
    --disk-size 300 \
    --machine-type ${MACHINE_TYPE} \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=wb-blinding-cabbage-2295 \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/autosomes_final_pruned.bed" \
    --input bim_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/autosomes_final_pruned.bim" \
    --input fam_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/autosomes_final_pruned.fam" \
    --input in_file="gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/updated_FID.fam" \
    --output-recursive output_folder="${output_folder}"

httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Job properties:
  job-id: clinvar-v8--nathani--260701-045710-23
  job-name: clinvar-v8
  user-id: nathani
Provider internal-id (operation): projects/wb-blinding-cabbage-2295/locations/us-central1/jobs/clinvar-v8--nathani--260701-045710-23-0-0
Launched job-id: clinvar-v8--nathani--260701-045710-23
To check the status, run:
  dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260701-045710-23' --users 'nathani' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260701-045710-23' --users 'nathani'


In [12]:
!dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260701-045710-23' --users 'nathani' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
clinvar-v8  SUCCESS   07-01 05:03:39

